# Search-o1 Reproduction

## 1. Environment Setup

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/search-o1-reproduction'
os.makedirs(f'{DRIVE_DIR}/outputs', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)

Mounted at /content/drive


In [ ]:
# Clone repo
%cd /content
if not os.path.exists('/content/Search-o1'):
    !git clone https://github.com/RUC-NLPIR/Search-o1.git
%cd /content/Search-o1

/content
Cloning into 'Search-o1'...
remote: Enumerating objects: 391, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 391 (delta 168), reused 156 (delta 156), pack-reused 211 (from 1)
Receiving objects: 100% (391/391), 4.51 MiB | 36.95 MiB/s, done.
Resolving deltas: 100% (195/195), done.
/content/Search-o1


In [ ]:
# Fixed installation: skip pyext, handle vllm separately
!pip install torch==2.5.1 transformers==4.46.1 sentencepiece==0.2.0 tqdm nltk
!pip install vllm==0.6.4
!pip install trafilatura requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 143.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 149.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 112.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.7/274.7 kB 35.0 MB/s eta 0:00:00


KeyboardInterrupt: 

## 2. First: Read the original bing_search.py
We need to understand the exact function signatures to create a drop-in replacement.

In [ ]:
%cd /content/Search-o1
!ls scripts/bing_search.py

/content/Search-o1
scripts/bing_search.py


In [ ]:
# Also check how run_search_o1.py calls these functions
# Focus on the lines that actually invoke search
!grep -n 'bing_web_search\|fetch_page_content\|extract_relevant\|extract_snippet' scripts/run_search_o1.py

17:    bing_web_search, 
18:    extract_relevant_info, 
19:    fetch_page_content, 
20:    extract_snippet_with_context
540:                                results = bing_web_search(search_query, bing_subscription_key, bing_endpoint, market='en-US', language='en')
549:                        relevant_info = extract_relevant_info(results)[:top_k]
619:                    fetched_contents = fetch_page_content(
640:                    success, filtered_context = extract_snippet_with_context(raw_context, doc_info['snippet'], context_chars=max_doc_len)


In [ ]:
# Also check run_naive_rag.py and run_rag_agent.py
print("--- run_naive_rag.py ---")
!grep -n 'bing_web_search\|fetch_page_content\|extract_relevant\|extract_snippet' scripts/run_naive_rag.py
print("\n--- run_rag_agent.py ---")
!grep -n 'bing_web_search\|fetch_page_content\|extract_relevant\|extract_snippet' scripts/run_rag_agent.py

--- run_naive_rag.py ---
10:    bing_web_search,
11:    extract_relevant_info,
12:    fetch_page_content,
13:    extract_snippet_with_context,
252:            results = bing_web_search(search_question, bing_subscription_key, bing_endpoint, market='en-US', language='en')
257:        relevant_info = extract_relevant_info(results)[:top_k]
279:    fetched_contents = fetch_page_content(
307:            success, context = extract_snippet_with_context(raw_context, snippet, context_chars=max_doc_len)

--- run_rag_agent.py ---
15:from bing_search import bing_web_search, extract_relevant_info, fetch_page_content
393:                            results = bing_web_search(query, bing_subscription_key, bing_endpoint, market='en-US', language='en')
400:                    relevant_info = extract_relevant_info(results)[:top_k]
451:                            contents = fetch_page_content(urls_to_fetch_filtered, use_jina=use_jina, jina_api_key=jina_api_key)


## 3. Set Brave API Key


In [ ]:
BRAVE_API_KEY = "USE_YOUR_OWN_API_KEY"

## 4. Test Brave Search API
Verify it returns good English results for science queries. (duckduckgo does not have good contents)

In [ ]:
import requests
import json

def brave_search(query, api_key, count=10):
    """Search using Brave Search API."""
    headers = {
        'Accept': 'application/json',
        'Accept-Encoding': 'gzip',
        'X-Subscription-Token': api_key
    }
    params = {
        'q': query,
        'count': count,
        'search_lang': 'en',
        'country': 'us'
    }
    resp = requests.get(
        'https://api.search.brave.com/res/v1/web/search',
        headers=headers,
        params=params,
        timeout=20
    )
    resp.raise_for_status()
    return resp.json()

# Test 1: The exact query from Search-o1 paper example
result = brave_search("structure of trans-cinnamaldehyde", BRAVE_API_KEY, count=5)

print("Search: 'structure of trans-cinnamaldehyde'")
print("=" * 60)
for i, r in enumerate(result.get('web', {}).get('results', [])):
    print(f"\n[{i+1}] {r['title']}")
    print(f"    URL: {r['url']}")
    print(f"    Snippet: {r.get('description', '')[:200]}")

Search: 'structure of trans-cinnamaldehyde'

[1] Cinnamaldehyde - Wikipedia
    URL: https://en.wikipedia.org/wiki/Cinnamaldehyde
    Snippet: The natural product is trans-cinnamaldehyde. The molecule consists of <strong>a benzene ring attached to an unsaturated aldehyde</strong>. Cinnamaldehyde is an α,β-unsaturated carbonyl compound.

[2] trans-Cinnamaldehyde - American ...
    URL: https://www.acs.org/molecule-of-the-week/archive/c/trans-cinnamaldehyde.html
    Snippet: ACS is one of the world’s largest scientific societies and the premier home of chemistry professionals. Find career opportunities, educational resources, professional development, and much more.

[3] trans-Cinnamaldehyde 99 14371-10-9
    URL: https://www.sigmaaldrich.com/US/en/product/aldrich/c80687
    Snippet: Aldrich-C80687; trans-Cinnamaldehyde 0.99; CAS No.: 14371-10-9; Synonyms: trans-3-Phenyl-2-propenal; <strong>Linear Formula: C6H5CH=CHCHO; Empirical Formula: C9H8O</strong>; find related products, pap

[4] t

In [ ]:
# Test 2: Another science query
result2 = brave_search("Grignard reagent reaction with aldehyde mechanism", BRAVE_API_KEY, count=5)

print("Search: 'Grignard reagent reaction with aldehyde mechanism'")
print("=" * 60)
for i, r in enumerate(result2.get('web', {}).get('results', [])):
    print(f"\n[{i+1}] {r['title']}")
    print(f"    Snippet: {r.get('description', '')[:200]}")

Search: 'Grignard reagent reaction with aldehyde mechanism'

[1] Grignard Reaction
    Snippet: The Grignard Reaction is the <strong>addition of an organomagnesium halide (Grignard reagent) to a ketone or aldehyde, to form a tertiary or secondary alcohol</strong>, respectively.

[2] Reactions of Grignard Reagents – Master Organic Chemistry
    Snippet: They will <strong>add to aldehydes and ketones to form alcohols</strong> (after a protonation step) They will add twice to esters to give tertiary alcohols. They will add to the less-substituted side 

[3] Grignard reagents in organic chemistry – Master Organic Chemistry
    Snippet: That means that carbon is more electron rich than magnesium and is actually nucleophilic! Here’s a closer look. In the reaction of Grignards with aldehydes, <strong>the carbon attacks the carbonyl car

[4] 3.4.2 – Grignard Reactions with Carbonyls – Organic Chemistry and Chemical Biology for the Students by the Students! (and the Profs…)
    Snippet: (The fu

In [ ]:
# Test 3: trafilatura page fetching with a real science URL
import trafilatura

test_url = result.get('web', {}).get('results', [{}])[0].get('url', '')
print(f"Fetching: {test_url}")
print("=" * 60)

downloaded = trafilatura.fetch_url(test_url)
if downloaded:
    text = trafilatura.extract(downloaded)
    if text:
        print(f"Extracted {len(text)} characters")
        print(f"\nFirst 500 chars:\n{text[:500]}")
    else:
        print("trafilatura couldn't extract text, using snippet instead")
else:
    print("Couldn't download page")

## 5. Create Drop-in Replacement for bing_search.py

This is the key step. We create a new `bing_search.py` that:
- Has the **exact same function signatures** as the original
- Uses Brave Search API instead of Bing
- Uses trafilatura instead of Jina Reader
- All other scripts (`run_search_o1.py`, `run_naive_rag.py`, `run_rag_agent.py`) don't need changes!

In [ ]:
%%writefile /content/Search-o1/scripts/bing_search.py

"""
Drop-in replacement for bing_search.py
Uses Brave Search API + snippet-based content
"""

import time
import requests
import re
from typing import Optional, List, Dict

def _truncate_query(query, max_len=200):
    """Clean and truncate query for Brave Search API."""
    for pattern in [r'\s*Choices:\s*', r'\s*\(A\)', r'\s*\ba\)', r'\s*Options:', r'\s*A\.']:
        parts = re.split(pattern, query, maxsplit=1)
        query = parts[0]
    query = re.sub(r'\s+', ' ', query).strip()
    if len(query) > max_len:
        sentences = re.split(r'[.?!]\s+', query)
        query = sentences[0]
        if len(query) > max_len:
            query = query[:max_len]
    return query

def bing_web_search(query, subscription_key, endpoint, market='en-US', language='en', timeout=20):
    query = _truncate_query(query)
    headers = {
        'Accept': 'application/json',
        'Accept-Encoding': 'gzip',
        'X-Subscription-Token': subscription_key
    }
    params = {
        'q': query,
        'count': 10,
        'search_lang': 'en',
        'country': 'us'
    }
    for attempt in range(3):
        try:
            resp = requests.get(
                'https://api.search.brave.com/res/v1/web/search',
                headers=headers, params=params, timeout=timeout
            )
            resp.raise_for_status()
            brave_data = resp.json()
            bing_compatible = {'webPages': {'value': []}}
            for r in brave_data.get('web', {}).get('results', []):
                bing_compatible['webPages']['value'].append({
                    'name': r.get('title', ''),
                    'url': r.get('url', ''),
                    'snippet': r.get('description', ''),
                })
            return bing_compatible
        except Exception as e:
            print(f"[Brave] Error attempt {attempt+1}: {e}")
            time.sleep(3 * (attempt + 1))
    return {'webPages': {'value': []}}


def extract_relevant_info(search_results):
    if not search_results or 'webPages' not in search_results:
        return []
    return [
        {'title': item.get('name',''), 'url': item.get('url',''), 'snippet': item.get('snippet','')}
        for item in search_results['webPages'].get('value', [])
    ]


def fetch_page_content(urls, max_workers=8, use_jina=False, jina_api_key=None, snippets: Optional[dict] = None):
    if isinstance(urls, str):
        urls = [urls]
    contents = {}
    for url in urls:
        if snippets and url in snippets:
            contents[url] = snippets[url]
    return contents


def extract_snippet_with_context(raw_context, snippet=None, context_chars=3000, max_results=10):
    """
    Compatible with BOTH calling patterns:
    1. run_naive_rag.py: extract_snippet_with_context(raw_context, snippet, context_chars=max_doc_len)
       - Returns (success: bool, context: str)
    2. run_search_o1.py: extract_snippet_with_context(search_results, max_results=10)
       - Returns dict of {url: snippet}
    """
    # Pattern 1: called from run_naive_rag.py with (raw_context, snippet, context_chars)
    if isinstance(raw_context, str):
        if snippet and snippet in raw_context:
            # Find snippet in raw context and extract surrounding text
            idx = raw_context.find(snippet)
            start = max(0, idx - context_chars // 2)
            end = min(len(raw_context), idx + len(snippet) + context_chars // 2)
            return True, raw_context[start:end]
        elif raw_context:
            return True, raw_context[:context_chars]
        elif snippet:
            return True, snippet
        else:
            return False, ""

    # Pattern 2: called from run_search_o1.py with (search_results, max_results)
    if isinstance(raw_context, dict):
        snippets = {}
        if 'webPages' not in raw_context:
            return snippets
        for item in raw_context['webPages'].get('value', [])[:max_results]:
            url = item.get('url', '')
            snip = item.get('snippet', '')
            if url and snip:
                snippets[url] = snip
        return snippets

    return False, ""

Overwriting /content/Search-o1/scripts/bing_search.py


## 6. Verify the Drop-in Replacement Works
Import our new module and test it exactly how `run_search_o1.py` would use it.

In [ ]:
# Simulate how run_search_o1.py uses these functions
import sys
sys.path.insert(0, '/content/Search-o1/scripts')

# This import should now use OUR replacement
import importlib
import bing_search
importlib.reload(bing_search)

from bing_search import (
    bing_web_search,
    extract_relevant_info,
    fetch_page_content,
    extract_snippet_with_context
)

print("All 4 functions imported successfully!")

✅ All 4 functions imported successfully!


In [ ]:
# Test the full pipeline as Search-o1 would use it

# Step 1: Search (as in run_search_o1.py)
query = "structure of trans-cinnamaldehyde molecular formula"
search_results = bing_web_search(
    query=query,
    subscription_key=BRAVE_API_KEY,
    endpoint="ignored",  # Original code passes Bing endpoint, we ignore it
)

# Step 2: Extract info
relevant_info = extract_relevant_info(search_results)
print(f"Found {len(relevant_info)} results")
for i, info in enumerate(relevant_info[:3]):
    print(f"\n[{i+1}] {info['title']}")
    print(f"    URL: {info['url']}")
    print(f"    Snippet: {info['snippet'][:150]}...")

Found 10 results

[1] Cinnamaldehyde - Wikipedia
    URL: https://en.wikipedia.org/wiki/Cinnamaldehyde
    Snippet: Cinnamaldehyde is an organic compound with the formula <strong>C9H8O or · C6H5CH=CHCHO</strong>. Occurring naturally as predominantly the trans (E) is...

[2] Cinnamaldehyde | C9H8O | CID 637511 - PubChem
    URL: https://pubchem.ncbi.nlm.nih.gov/compound/Cinnamaldehyde
    Snippet: Cinnamaldehyde | C9H8O | CID 637511 - structure, chemical names, physical and chemical properties, classification, patents, literature, biological act...

[3] trans-Cinnamaldehyde 99 14371-10-9
    URL: https://www.sigmaaldrich.com/US/en/product/aldrich/c80687
    Snippet: Aldrich-C80687; trans-Cinnamaldehyde 0.99; CAS No.: 14371-10-9; Synonyms: trans-3-Phenyl-2-propenal; <strong>Linear Formula: C6H5CH=CHCHO</strong>; Em...


In [ ]:
# Step 3: Extract snippets
snippets = extract_snippet_with_context(search_results)
print(f"Got snippets for {len(snippets)} URLs")

# Step 4: Fetch full page content
urls = list(snippets.keys())[:3]  # Only fetch first 3 to save time
print(f"\nFetching content for {len(urls)} pages...")
page_contents = fetch_page_content(
    urls=urls,
    use_jina=False,  # Ignored in our replacement
    jina_api_key=None,
    snippets=snippets  # Fallback if page fetch fails
)

for url, content in page_contents.items():
    print(f"\n{'='*60}")
    print(f"URL: {url}")
    print(f"Content length: {len(content)} chars")
    print(f"Preview: {content[:300]}...")

In [ ]:
import importlib, sys
sys.path.insert(0, '/content/Search-o1/scripts')
import bing_search
importlib.reload(bing_search)
from bing_search import *

# Test
results = bing_web_search("structure of trans-cinnamaldehyde", BRAVE_API_KEY, "ignored")
snippets = extract_snippet_with_context(results)
contents = fetch_page_content(list(snippets.keys())[:3], snippets=snippets)

print(f"✅ Got {len(contents)} page contents (from snippets)")
for url, content in contents.items():
    print(f"\nURL: {url}")
    print(f"Content: {content[:200]}")

✅ Got 3 page contents (from snippets)

URL: https://en.wikipedia.org/wiki/Cinnamaldehyde
Content: The natural product is trans-cinnamaldehyde. The molecule consists of <strong>a benzene ring attached to an unsaturated aldehyde</strong>. Cinnamaldehyde is an α,β-unsaturated carbonyl compound.

URL: https://www.acs.org/molecule-of-the-week/archive/c/trans-cinnamaldehyde.html
Content: ACS is one of the world’s largest scientific societies and the premier home of chemistry professionals. Find career opportunities, educational resources, professional development, and much more.

URL: https://www.sigmaaldrich.com/US/en/product/aldrich/c80687
Content: Aldrich-C80687; trans-Cinnamaldehyde 0.99; CAS No.: 14371-10-9; Synonyms: trans-3-Phenyl-2-propenal; <strong>Linear Formula: C6H5CH=CHCHO; Empirical Formula: C9H8O</strong>; find related products, pap


## 7. Quick Integration Test
Verify that `run_search_o1.py` can at least parse arguments with our replacement.

# Experiments Part

## 1. Prepare GPQA Diamond Dataset

GPQA is a gated dataset on HuggingFace. Two options:
- Option A: Accept terms on HF and use `huggingface-cli login`
- Option B: Use the public mirror `fingertap/GPQA-Diamond`


In [ ]:
from datasets import load_dataset

# Try public mirror first
try:
    gpqa = load_dataset('fingertap/GPQA-Diamond', split='test')
    print(f'✅ Loaded GPQA Diamond from public mirror: {len(gpqa)} questions')
    print(f'Columns: {gpqa.column_names}')
    print(f'\nFirst example preview:')
    print(json.dumps({k: str(v)[:100] for k, v in gpqa[0].items()}, indent=2))
except Exception as e:
    print(f'Mirror failed: {e}')
    print('\n⚠️ Try Option A: Go to https://huggingface.co/datasets/Idavidrein/gpqa')
    print('   Accept terms, then run: huggingface-cli login')
    print('   Then: gpqa = load_dataset("Idavidrein/gpqa", "gpqa_diamond", split="train")')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


test/gpqa_diamond.parquet:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/198 [00:00<?, ? examples/s]

✅ Loaded GPQA Diamond from public mirror: 198 questions
Columns: ['question', 'answer']

First example preview:
{
  "question": "Among the following exoplanets, which one has the highest density?\n\na) An Earth-mass and Earth-radiu",
  "answer": "D"
}


In [ ]:
# Examine the data format to understand how to convert it
print("All column names:")
print(gpqa.column_names)
print(f"\nTotal questions: {len(gpqa)}")

# Show a full example
example = gpqa[0]
print("\n" + "="*60)
print("FULL FIRST EXAMPLE:")
print("="*60)
for k, v in example.items():
    print(f"\n--- {k} ---")
    print(str(v)[:300])

All column names:
['question', 'answer']

Total questions: 198

FULL FIRST EXAMPLE:

--- question ---
Among the following exoplanets, which one has the highest density?

a) An Earth-mass and Earth-radius planet.
b) A planet with 2 Earth masses and a density of approximately 5.5 g/cm^3.
c) A planet with the same composition as Earth but 5 times more massive than Earth.
d) A planet with the same compo

--- answer ---
D


## 2. Convert GPQA to Search-o1 Format

Based on the README, multi-choice tasks need: `{'Question': str, 'Correct Choice': str}`

Run the cells above first to see the exact column names, then we'll convert.

In [ ]:
!huggingface-cli login

In [ ]:
# ============================================
# GPQA Data Preparation (matching author's exact format)
# ============================================
import json, random, os

# Download GPQA from HuggingFace
# Method 1: Try public dataset
try:
    from datasets import load_dataset
    ds = load_dataset("Idavidrein/gpqa", "gpqa_diamond", split="train",
                      trust_remote_code=True)
    print(f"✅ Loaded from HF: {len(ds)} questions")
except Exception as e:
    print(f"⚠️ Gated dataset, need login: {e}")
    print("Run: !huggingface-cli login")
    print("Then re-run this cell")
    ds = None

if ds is not None:
    filtered_data = []
    for idx, row in enumerate(ds):
        # Extract answers and shuffle (same as author's code)
        answers = [
            ('Correct Answer', row['Correct Answer']),
            ('Incorrect Answer 1', row['Incorrect Answer 1']),
            ('Incorrect Answer 2', row['Incorrect Answer 2']),
            ('Incorrect Answer 3', row['Incorrect Answer 3']),
        ]
        random.shuffle(answers)

        # Assign A, B, C, D
        choices = ['A', 'B', 'C', 'D']
        formatted_answers = []
        correct_choice = None
        for i, (label, answer) in enumerate(answers):
            choice = choices[i]
            formatted_answers.append((choice, answer))
            if label == 'Correct Answer':
                correct_choice = choice

        formatted_choices = "\n".join([f"({c}) {a}" for c, a in formatted_answers])
        question_with_choices = f"{row['Question']} Choices:\n{formatted_choices}\n"

        filtered_data.append({
            'id': idx,
            'Question': question_with_choices,
            'Subdomain': row.get('Subdomain', ''),
            'High-level domain': row.get('High-level domain', ''),
            'Correct Answer': row['Correct Answer'],
            'Incorrect Answer 1': row['Incorrect Answer 1'],
            'Incorrect Answer 2': row['Incorrect Answer 2'],
            'Incorrect Answer 3': row['Incorrect Answer 3'],
            'Correct Choice': correct_choice,
        })

    # Save
    os.makedirs('data/GPQA', exist_ok=True)
    output_path = 'data/GPQA/diamond.json'
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(filtered_data, f, indent=4, ensure_ascii=False)

    print(f"✅ Saved {len(filtered_data)} questions to {output_path}")
    print(f"\nExample:")
    print(f"Question: {filtered_data[0]['Question'][:200]}...")
    print(f"Correct Choice: {filtered_data[0]['Correct Choice']}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Idavidrein/gpqa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Idavidrein/gpqa' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


gpqa_diamond.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/198 [00:00<?, ? examples/s]

✅ Loaded from HF: 198 questions
✅ Saved 198 questions to data/GPQA/diamond.json

Example:
Question: Two quantum states with energies E1 and E2 have a lifetime of 10^-9 sec and 10^-8 sec, respectively. We want to clearly distinguish these two energy levels. Which one of the following options could be...
Correct Choice: B


In [ ]:
!huggingface-cli login --token YOUR_HF_TOKEN

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `colab2` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `colab2`


In [ ]:
%cd /content/Search-o1
!grep -n 'data_path\|GPQA\|gpqa.*json\|diamond' scripts/run_direct_gen.py | head -20

/content/Search-o1
35:        choices=['test', 'diamond', 'main', 'extended'],
109:        data_path = f'./data/MATH500/{split}.json'
111:        data_path = f'./data/GPQA/{split}.json'
113:        data_path = f'./data/AIME/{split}.json'
115:        data_path = f'./data/AMC/{split}.json'
117:        data_path = f'./data/LiveCodeBench/{split}.json'
119:        data_path = f'./data/QA_Datasets/{dataset_name}.json'
159:    with open(data_path, mode='r', encoding='utf-8') as json_file:


In [ ]:
import os
DRIVE_DIR = '/content/drive/MyDrive/search-o1-reproduction'
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

print(f"Downloading tokenizer for {MODEL_NAME}...")
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer loaded")

test_tokens = tokenizer.encode("Hello, this is a test.")
print(f"✅ Tokenizer test: {len(test_tokens)} tokens")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded
✅ Tokenizer test: 8 tokens


In [ ]:
# Fix: make pyext importable as a dummy module
!mkdir -p /usr/local/lib/python3.12/dist-packages/pyext
!echo "class RuntimeModule: pass" > /usr/local/lib/python3.12/dist-packages/pyext/__init__.py

# Verify fix
!python -c "from pyext import RuntimeModule; print('✅ pyext fix works')"

✅ pyext fix works


In [ ]:
%cd /content/Search-o1/scripts

!python run_direct_gen.py \
    --dataset_name gpqa \
    --split diamond \
    --subset_num 3 \
    --model_path deepseek-ai/DeepSeek-R1-Distill-Qwen-7B \
    --temperature 0.7 \
    --top_p 0.8 \
    --top_k 20 \
    --repetition_penalty 1.05 \
    --max_tokens 16384

/content/Search-o1/scripts
2026-03-06 06:02:54.909715: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772776974.931421   10043 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772776974.938100   10043 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772776974.954711   10043 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772776974.954739   10043 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772776974.954742   10043 computation_placer.cc:

In [ ]:
!ls -la outputs/gpqa.ds-qwen-7b.direct/

total 76
drwxr-xr-x 2 root root  4096 Mar  6 06:06 .
drwxr-xr-x 3 root root  4096 Mar  6 06:03 ..
-rw-r--r-- 1 root root 62723 Mar  6 06:06 diamond.3.6,6:6.json
-rw-r--r-- 1 root root   561 Mar  6 06:06 diamond.3.6,6:6.metrics.json


In [ ]:
# Then read the actual file
import glob, json

files = glob.glob('outputs/gpqa.ds-qwen-7b.direct/*')
print("Files:", files)

for f in files:
    print(f"\n{'='*60}")
    print(f"File: {f}")
    with open(f) as fh:
        content = fh.read()
    print(f"Size: {len(content)} chars")
    print(f"Preview:\n{content[:2000]}")

Files: ['outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:6.metrics.json', 'outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:6.json']

File: outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:6.metrics.json
Size: 621 chars
Preview:
{
    "overall": {
        "em": 0.6666666666666666,
        "acc": 0.6666666666666666,
        "f1": 0.6666666666666666,
        "math_equal": 0.6666666666666666,
        "num_valid_answer": "3 of 3",
        "query_latency": "52267 ms"
    },
    "per_domain": {
        "Physics": {
            "em": 1.0,
            "acc": 1.0,
            "f1": 1.0,
            "math_equal": 1.0,
            "num_valid_answer": "2 of 2"
        },
        "Chemistry": {
            "em": 0.0,
            "acc": 0.0,
            "f1": 0.0,
            "math_equal": 0.0,
            "num_valid_answer": "1 of 1"
        }
    }
}

File: outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:6.json
Size: 61432 chars
Preview:
[
    {
        "id": 0,
        "Question": "<｜begin▁of▁sentence｜><｜User｜

In [ ]:
import json

with open('outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:6.json') as f:
    data = json.load(f)

# Check the end of each output to see if \boxed{} is there
for i, item in enumerate(data):
    output = item['Output']
    # Show last 500 chars
    print(f"\n{'='*60}")
    print(f"Question {i} (Correct: {item['Correct Choice']})")
    print(f"Output ending:")
    print(output[-500:])

    # Check for boxed answer
    import re
    boxed = re.findall(r'\\boxed\{(.*?)\}', output)
    print(f"\nExtracted boxed answers: {boxed}")


Question 0 (Correct: B)
Output ending:
, so the energy difference is sufficient to resolve the states.  
- **(C) \( 10^{-11} \, \text{eV} \):** This is smaller than \( 6.58 \times 10^{-7} \, \text{eV} \), so the energy difference is insufficient to resolve the states.  
- **(D) \( 10^{-9} \, \text{eV} \):** This is smaller than \( 6.58 \times 10^{-7} \, \text{eV} \), so the energy difference is insufficient to resolve the states.

Thus, the only option that provides a clear resolution is **(B) \( 10^{-4} \, \text{eV} \)**.

ANSWER: B

Extracted boxed answers: []

Question 1 (Correct: A)
Output ending:
This would require the addition of an extra carbon atom, which is not supported by the reactions described.  
- **Option B (10)**: This matches the number of carbon atoms in the ketone intermediate after oxidation and the product after the Horner-Wadsworth-Emmons reaction.  
- **Option C (12)** and **Option D (14)**: These would require significant structural changes or additions that ar

In [ ]:
# Fix evaluate.py to also extract "ANSWER: X" format
import re

# Read current evaluate.py
with open('/content/Search-o1/scripts/evaluate.py', 'r') as f:
    content = f.read()

# Find the extract_answer function and add ANSWER: pattern
old_pattern = """        pattern = r'\\\\boxed\\{(.*)\\}'
        matches = re.findall(pattern, output)
        if matches:
            extracted_text = matches[-1]  # Take the last match"""

new_pattern = """        # Try \\boxed{} first
        pattern = r'\\\\boxed\\{(.*)\\}'
        matches = re.findall(pattern, output)
        if matches:
            extracted_text = matches[-1]  # Take the last match
        else:
            # Fallback: try "ANSWER: X" format (common in DeepSeek-R1)
            answer_pattern = r'ANSWER:\\s*([A-D])'
            answer_matches = re.findall(answer_pattern, output)
            if answer_matches:
                extracted_text = answer_matches[-1]"""

if old_pattern in content:
    content = content.replace(old_pattern, new_pattern)
    with open('/content/Search-o1/scripts/evaluate.py', 'w') as f:
        f.write(content)
    print("✅ evaluate.py patched to support ANSWER: format")
else:
    print("⚠️ Couldn't find exact pattern. Let me show the relevant section:")
    # Show the function for manual fix
    for i, line in enumerate(content.split('\n')):
        if 'boxed' in line or 'extract_answer' in line:
            print(f"  Line {i+1}: {line}")

✅ evaluate.py patched to support ANSWER: format


In [ ]:
!python evaluate.py --output_path outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:6.json

Evaluation completed. Metrics saved to outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:6.metrics.json


In [ ]:
%cd /content/Search-o1/scripts

!python run_direct_gen.py \
    --dataset_name gpqa \
    --split diamond \
    --model_path deepseek-ai/DeepSeek-R1-Distill-Qwen-7B \
    --temperature 0.7 \
    --top_p 0.8 \
    --top_k 20 \
    --repetition_penalty 1.05 \
    --max_tokens 16384

/content/Search-o1/scripts
2026-03-06 06:10:23.061151: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772777423.083439   12148 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772777423.090144   12148 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772777423.106934   12148 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772777423.106963   12148 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772777423.106966   12148 computation_placer.cc:

In [ ]:
!cat outputs/gpqa.ds-qwen-7b.direct/*.metrics.json

{
    "overall": {
        "em": 0.46464646464646464,
        "acc": 0.46464646464646464,
        "f1": 0.46464646464646464,
        "math_equal": 0.46464646464646464,
        "num_valid_answer": "196 of 198",
        "query_latency": "3919 ms"
    },
    "per_domain": {
        "Physics": {
            "em": 0.6511627906976745,
            "acc": 0.6511627906976745,
            "f1": 0.6511627906976745,
            "math_equal": 0.6511627906976745,
            "num_valid_answer": "85 of 86"
        },
        "Chemistry": {
            "em": 0.3010752688172043,
            "acc": 0.3010752688172043,
            "f1": 0.3010752688172043,
            "math_equal": 0.3010752688172043,
            "num_valid_answer": "92 of 93"
        },
        "Biology": {
            "em": 0.42105263157894735,
            "acc": 0.42105263157894735,
            "f1": 0.42105263157894735,
            "math_equal": 0.42105263157894735,
            "num_valid_answer": "19 of 19"
        }
    }
}{
    "o

In [ ]:
!cp -r outputs/gpqa.ds-qwen-7b.direct /content/drive/MyDrive/search-o1-reproduction/outputs/

In [ ]:
!python run_naive_rag.py \
    --dataset_name gpqa \
    --split diamond \
    --model_path deepseek-ai/DeepSeek-R1-Distill-Qwen-7B \
    --bing_subscription_key {BRAVE_KEY} \
    --top_k 10 \
    --max_doc_len 3000 \
    --temperature 0.7 \
    --top_p 0.8 \
    --repetition_penalty 1.05 \
    --max_tokens 16384

2026-03-06 06:42:33.035710: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772779353.057125   20782 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772779353.063644   20782 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772779353.080210   20782 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772779353.080235   20782 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772779353.080238   20782 computation_placer.cc:177] computation placer alr

In [ ]:
!cat outputs/gpqa.ds-qwen-7b.naive_rag/*.metrics.json

cat: 'outputs/gpqa.ds-qwen-7b.naive_rag/*.metrics.json': No such file or directory


In [ ]:
!find outputs/ -name "*.metrics.json" -type f

outputs/runs.baselines/gpqa.deepseek-r1-distill-qwen-7b.naive_rag/diamond.3.6,6:58.metrics.json
outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:24.metrics.json
outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:6.metrics.json


In [ ]:
!find outputs/ -type f | head -20

outputs/runs.baselines/gpqa.deepseek-r1-distill-qwen-7b.naive_rag/diamond.3.6,6:58.json
outputs/runs.baselines/gpqa.deepseek-r1-distill-qwen-7b.naive_rag/diamond.3.6,6:58.metrics.json
outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:24.metrics.json
outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:24.json
outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:6.metrics.json
outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:6.json


In [ ]:
!cat outputs/runs.baselines/gpqa.deepseek-r1-distill-qwen-7b.naive_rag/diamond.3.6,6:58.metrics.json

{
    "overall": {
        "em": 0.5,
        "acc": 0.5,
        "f1": 0.5,
        "math_equal": 0.5,
        "num_valid_answer": "188 of 198",
        "query_latency": "4412 ms"
    },
    "per_domain": {
        "Physics": {
            "em": 0.7093023255813954,
            "acc": 0.7093023255813954,
            "f1": 0.7093023255813954,
            "math_equal": 0.7093023255813954,
            "num_valid_answer": "83 of 86"
        },
        "Chemistry": {
            "em": 0.3118279569892473,
            "acc": 0.3118279569892473,
            "f1": 0.3118279569892473,
            "math_equal": 0.3118279569892473,
            "num_valid_answer": "88 of 93"
        },
        "Biology": {
            "em": 0.47368421052631576,
            "acc": 0.47368421052631576,
            "f1": 0.47368421052631576,
            "math_equal": 0.47368421052631576,
            "num_valid_answer": "17 of 19"
        }
    }
}

In [ ]:
# Backup
!cp -r outputs/ /content/drive/MyDrive/search-o1-reproduction/outputs_gpqa/


In [ ]:
!python run_rag_agent.py \
    --dataset_name gpqa \
    --split diamond \
    --model_path deepseek-ai/DeepSeek-R1-Distill-Qwen-7B \
    --bing_subscription_key {BRAVE_KEY} \
    --max_search_limit 5 \
    --max_url_fetch 5 \
    --max_turn 10 \
    --top_k 10 \
    --temperature 0.7 \
    --top_p 0.8 \
    --repetition_penalty 1.05 \
    --max_tokens 16384

2026-03-06 07:04:57.292831: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772780697.314776   26770 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772780697.321489   26770 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772780697.338378   26770 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772780697.338420   26770 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772780697.338423   26770 computation_placer.cc:177] computation placer alr

In [ ]:
# GPQA - Search-o1
!python run_search_o1.py \
    --dataset_name gpqa \
    --split diamond \
    --model_path deepseek-ai/DeepSeek-R1-Distill-Qwen-7B \
    --bing_subscription_key $BRAVE_KEY \
    --max_search_limit 5 \
    --max_turn 10 \
    --top_k 10 \
    --max_doc_len 3000 \
    --temperature 0.7 \
    --top_p 0.8 \
    --repetition_penalty 1.05 \
    --max_tokens 16384

In [ ]:
import json
with open('outputs/runs.baselines/gpqa.deepseek-r1-distill-qwen-7b.search_o1/diamond.3.6,7:26.json') as f:
    data = json.load(f)

# Check how many questions triggered search
search_count = 0
for item in data:
    output = item.get('Output', '')
    if '<|begin_search_query|>' in output or 'begin_search_query' in output:
        search_count += 1

print(f"Questions that triggered search: {search_count} / {len(data)}")

# Show first example's output ending
print(f"\nFirst example output (last 500 chars):")
print(data[0]['Output'][-500:])

Questions that triggered search: 0 / 198

First example output (last 500 chars):
 option.
</think>

The energy difference between the two quantum states must be larger than the combined uncertainty of their natural linewidths to clearly resolve them. Using the uncertainty principle, the energy uncertainty for each state is calculated as ΔE ≈ ħ / τ. Adding the uncertainties for both states, the combined uncertainty is approximately 7.24e-7 eV. Therefore, the energy difference must be greater than this value. Among the options, only 1e-4 eV satisfies this condition.

\boxed{B}


In [ ]:
import json
with open('outputs/runs.baselines/gpqa.deepseek-r1-distill-qwen-7b.search_o1/diamond.3.6,7:26.json') as f:
    data = json.load(f)

# Show the full Question field (which includes the prompt)
print("Full input to model:")
print(data[0]['Question'][:1500])

Full input to model:
<｜begin▁of▁sentence｜><｜User｜>You are a reasoning assistant with the ability to perform web searches to help you answer the user's question accurately. You have special tools:

- To perform a search: write <|begin_search_query|> your query here <|end_search_query|>.
Then, the system will search and analyze relevant web pages, then provide you with helpful information in the format <|begin_search_result|> ...search results... <|end_search_result|>.

You can repeat the search process multiple times if necessary. The maximum number of search attempts is limited to 5.

Once you have all the information you need, continue your reasoning.

Example:
Question: "What is the energy range of pp III neutrinos?"
Assistant thinking steps:
- I might need to look up details about pp III neutrinos.

Assistant:
<|begin_search_query|>pp III neutrino energy spectrum<|end_search_query|>

(System returns processed information from relevant web pages)

Assistant continues reasoning with t

In [ ]:
import json
with open('outputs/runs.baselines/gpqa.deepseek-r1-distill-qwen-7b.search_o1/diamond.3.6,7:26.json') as f:
    data = json.load(f)

# Show beginning of output for first 3 questions
for i in range(3):
    print(f"\n{'='*60}")
    print(f"Q{i} output start (first 300 chars):")
    print(data[i]['Output'][:300])


Q0 output start (first 300 chars):
Okay, so I've got this physics problem here about quantum states and their lifetimes. Hmm, let me try to figure this out step by step. Alright, the question says we have two quantum states with energies E1 and E2, and their lifetimes are 10^-9 seconds and 10^-8 seconds respectively. We need to deter

Q1 output start (first 300 chars):
Okay, so I'm trying to figure out how many carbon atoms are in product 3 after these reactions. Let me break this down step by step because I'm a bit confused about the organic chemistry reactions involved.

First, the starting material is trans-cinnamaldehyde. I remember that cinnamaldehyde has a b

Q2 output start (first 300 chars):
Okay, so I've got this quantum mechanics problem here, and I'm trying to figure it out step by step. Let me start by reading the question carefully.

The problem says that there's a spin-half particle in a specific state, which is a linear combination of spin-up and spin-down states. The s

In [ ]:
!sed -n '1,30p' run_search_o1.py

# run_search_o1.py
import os
import json
import time
import re
from tqdm import tqdm
import numpy as np
import torch
import string
from typing import Optional, Tuple, List, Dict
import argparse

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

from bing_search import (
    bing_web_search, 
    extract_relevant_info, 
    fetch_page_content, 
    extract_snippet_with_context
)
from evaluate import (
    run_evaluation, 
    extract_answer
)
from prompts import (
    get_gpqa_search_o1_instruction, 
    get_math_search_o1_instruction, 
    get_code_search_o1_instruction, 
    get_singleqa_search_o1_instruction, 


In [ ]:
# Look at how the generation loop works - especially the search detection
!grep -n 'begin_search_query\|end_search_query\|search_query\|generate\|vllm\|sampling' run_search_o1.py | head -30

14:from vllm import LLM, SamplingParams
40:BEGIN_SEARCH_QUERY = "<|begin_search_query|>"
41:END_SEARCH_QUERY = "<|end_search_query|>"
135:        help="Top-p sampling parameter."
139:        '--top_k_sampling',
142:        help="Top-k sampling parameter."
156:        help="Maximum number of tokens to generate. If not set, defaults based on the model and dataset."
190:    top_k_sampling = args.top_k_sampling
286:    def generate_webpage_to_reasonchain_batch(
304:        output = llm.generate(
306:            sampling_params=SamplingParams(
399:        sampling_params = SamplingParams(
403:            top_k=top_k_sampling,
408:        output_list = llm.generate(prompts, sampling_params=sampling_params)
524:                # Append generated text to prompt and output
529:                search_query = extract_between(text, BEGIN_SEARCH_QUERY, END_SEARCH_QUERY)
532:                if search_query and seq['output'].rstrip().endswith(END_SEARCH_QUERY):
533:                    if seq['search_

In [ ]:
# Look at the prompt construction part
!sed -n '180,250p' run_search_o1.py

    dataset_name = args.dataset_name
    split = args.split
    subset_num = args.subset_num
    MAX_SEARCH_LIMIT = args.max_search_limit
    MAX_TURN = args.max_turn
    top_k = args.top_k
    max_doc_len = args.max_doc_len
    model_path = args.model_path
    temperature = args.temperature
    top_p = args.top_p
    top_k_sampling = args.top_k_sampling
    repetition_penalty = args.repetition_penalty
    max_tokens = args.max_tokens
    bing_subscription_key = args.bing_subscription_key
    bing_endpoint = args.bing_endpoint
    use_jina = args.use_jina
    jina_api_key = args.jina_api_key
    
    # Adjust parameters based on dataset
    if dataset_name in ['nq', 'triviaqa', 'hotpotqa', 'musique', 'bamboogle', '2wiki', 'medmcqa', 'pubhealth']:
        MAX_SEARCH_LIMIT = 5
        if dataset_name in ['hotpotqa', 'musique', 'bamboogle', '2wiki']:
            MAX_SEARCH_LIMIT = 10
            MAX_TURN = 15
        top_k = 10
        max_doc_len = 3000
    
    if args.jina_api_key == '

In [ ]:
# ============================================
# Prepare NQ and HotpotQA datasets
# ============================================
%cd /content/Search-o1/scripts

import json, os

os.makedirs('data/QA_Datasets', exist_ok=True)

# --- HotpotQA ---
print("Downloading HotpotQA...")
from datasets import load_dataset
hotpot = load_dataset("hotpot_qa", "fullwiki", split="validation")
hotpot_data = []
for i, item in enumerate(hotpot):
    if i >= 500: break
    hotpot_data.append({
        'id': i,
        'Question': item['question'],
        'answer': [item['answer']]
    })
with open('data/QA_Datasets/hotpotqa.json', 'w') as f:
    json.dump(hotpot_data, f, indent=4, ensure_ascii=False)
print(f"✅ HotpotQA: {len(hotpot_data)} questions")

# --- NQ (Natural Questions) ---
# NQ is large, use FlashRAG version which is pre-processed
print("\nDownloading NQ...")
try:
    nq = load_dataset("RUC-NLPIR/FlashRAG_datasets", "nq", split="test")
    nq_data = []
    for i, item in enumerate(nq):
        if i >= 500: break
        nq_data.append({
            'id': i,
            'Question': item['question'],
            'answer': item['golden_answers'] if isinstance(item['golden_answers'], list) else [item['golden_answers']]
        })
    with open('data/QA_Datasets/nq.json', 'w') as f:
        json.dump(nq_data, f, indent=4, ensure_ascii=False)
    print(f"✅ NQ: {len(nq_data)} questions")
except Exception as e:
    print(f"FlashRAG NQ failed: {e}")
    print("Trying alternative...")
    nq = load_dataset("google-research-datasets/nq_open", split="validation")
    nq_data = []
    for i, item in enumerate(nq):
        if i >= 500: break
        nq_data.append({
            'id': i,
            'Question': item['question'],
            'answer': item['answer'] if isinstance(item['answer'], list) else [item['answer']]
        })
    with open('data/QA_Datasets/nq.json', 'w') as f:
        json.dump(nq_data, f, indent=4, ensure_ascii=False)
    print(f"✅ NQ: {len(nq_data)} questions")

print("\n✅ All datasets ready!")
print(f"   HotpotQA: data/QA_Datasets/hotpotqa.json")
print(f"   NQ: data/QA_Datasets/nq.json")

/content/Search-o1/scripts


README.md: 0.00B [00:00, ?B/s]

fullwiki/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

fullwiki/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

fullwiki/validation-00000-of-00001.parqu(…):   0%|          | 0.00/28.0M [00:00<?, ?B/s]

...

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

✅ NQ: 500 questions

✅ All datasets ready!
   HotpotQA: data/QA_Datasets/hotpotqa.json
   NQ: data/QA_Datasets/nq.json


In [ ]:
import os
os.environ['MODEL'] = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
os.environ['BRAVE_KEY'] = "USE_YOUR_OWN_API_KEY_HERE"

%cd /content/Search-o1/scripts
print("✅ Ready")

/content/Search-o1/scripts
✅ Ready


In [ ]:
!cp -r outputs/ /content/drive/MyDrive/search-o1-reproduction/outputs_all/

In [ ]:
# NQ - Direct Reasoning
!python run_direct_gen.py \
    --dataset_name nq --split test \
    --model_path $MODEL \
    --temperature 0.7 --top_p 0.8 --top_k 20 \
    --repetition_penalty 1.05 --max_tokens 16384

In [ ]:
# NQ - Standard RAG
!python run_naive_rag.py \
    --dataset_name nq --split test \
    --model_path $MODEL \
    --bing_subscription_key $BRAVE_KEY \
    --top_k 10 --max_doc_len 3000 \
    --temperature 0.7 --top_p 0.8 \
    --repetition_penalty 1.05 --max_tokens 16384

2026-03-06 08:12:14.253491: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 08:12:14.271894: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772784734.293737   45128 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772784734.300446   45128 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772784734.317396   45128 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!cp -r outputs/ /content/drive/MyDrive/search-o1-reproduction/outputs_all/

In [ ]:
# NQ - RAgent
!python run_rag_agent.py \
    --dataset_name nq --split test \
    --model_path $MODEL \
    --bing_subscription_key $BRAVE_KEY \
    --max_search_limit 5 --max_url_fetch 5 --max_turn 10 --top_k 10 \
    --temperature 0.7 --top_p 0.8 \
    --repetition_penalty 1.05 --max_tokens 16384

2026-03-06 08:25:09.795728: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 08:25:09.813598: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772785509.834867   48676 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772785509.841275   48676 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772785509.857793   48676 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!cp -r outputs/ /content/drive/MyDrive/search-o1-reproduction/outputs_all/

In [ ]:
# NQ - Search-o1
!python run_search_o1.py \
    --dataset_name nq --split test \
    --model_path $MODEL \
    --bing_subscription_key $BRAVE_KEY \
    --max_search_limit 5 --max_turn 10 --top_k 10 --max_doc_len 3000 \
    --temperature 0.7 --top_p 0.8 \
    --repetition_penalty 1.05 --max_tokens 16384

2026-03-06 08:29:25.889090: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 08:29:25.906690: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772785765.927621   49936 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772785765.934007   49936 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772785765.950464   49936 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!cp -r outputs/ /content/drive/MyDrive/search-o1-reproduction/outputs_all/

In [ ]:
# HotpotQA - Direct Reasoning
!python run_direct_gen.py \
    --dataset_name hotpotqa --split test \
    --model_path $MODEL \
    --temperature 0.7 --top_p 0.8 --top_k 20 \
    --repetition_penalty 1.05 --max_tokens 16384

2026-03-06 08:31:58.318283: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 08:31:58.336177: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772785918.357134   50705 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772785918.363756   50705 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772785918.380367   50705 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!cp -r outputs/ /content/drive/MyDrive/search-o1-reproduction/outputs_all/

In [ ]:
# HotpotQA - Standard RAG
!python run_naive_rag.py \
    --dataset_name hotpotqa --split test \
    --model_path $MODEL \
    --bing_subscription_key $BRAVE_KEY \
    --top_k 10 --max_doc_len 3000 \
    --temperature 0.7 --top_p 0.8 \
    --repetition_penalty 1.05 --max_tokens 16384

2026-03-06 08:39:01.579193: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 08:39:01.596625: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772786341.617638   52671 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772786341.624091   52671 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772786341.640379   52671 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!cp -r outputs/ /content/drive/MyDrive/search-o1-reproduction/outputs_all/

In [ ]:
# HotpotQA - RAgent
!python run_rag_agent.py \
    --dataset_name hotpotqa --split test \
    --model_path $MODEL \
    --bing_subscription_key $BRAVE_KEY \
    --max_search_limit 10 --max_url_fetch 5 --max_turn 15 --top_k 10 \
    --temperature 0.7 --top_p 0.8 \
    --repetition_penalty 1.05 --max_tokens 16384

2026-03-06 08:56:24.054737: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 08:56:24.072348: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772787384.093668   57390 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772787384.100147   57390 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772787384.116584   57390 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!cp -r outputs/ /content/drive/MyDrive/search-o1-reproduction/outputs_all/

In [ ]:
# HotpotQA - Search-o1
!python run_search_o1.py \
    --dataset_name hotpotqa --split test \
    --model_path $MODEL \
    --bing_subscription_key $BRAVE_KEY \
    --max_search_limit 10 --max_turn 15 --top_k 10 --max_doc_len 3000 \
    --temperature 0.7 --top_p 0.8 \
    --repetition_penalty 1.05 --max_tokens 16384

2026-03-06 09:01:16.527762: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-06 09:01:16.545941: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772787676.567360   58827 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772787676.573940   58827 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772787676.590616   58827 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!cp -r outputs/ /content/drive/MyDrive/search-o1-reproduction/outputs_all/

In [ ]:
import json, glob

for f in sorted(glob.glob('outputs/**/*.metrics.json', recursive=True)):
    with open(f) as fh:
        data = json.load(fh)
    overall = data['overall']
    print(f"{f}")
    print(f"  em={overall['em']:.3f}  f1={overall.get('f1', 0):.3f}  valid={overall['num_valid_answer']}")
    print()

outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:24.metrics.json
  em=0.465  f1=0.465  valid=196 of 198

outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:6.metrics.json
  em=0.667  f1=0.667  valid=3 of 3

outputs/runs.baselines/gpqa.deepseek-r1-distill-qwen-7b.naive_rag/diamond.3.6,6:58.metrics.json
  em=0.500  f1=0.500  valid=188 of 198

outputs/runs.baselines/gpqa.deepseek-r1-distill-qwen-7b.rag_agent/diamond.3.6,7:12.metrics.json
  em=0.439  f1=0.439  valid=167 of 198

outputs/runs.baselines/gpqa.deepseek-r1-distill-qwen-7b.search_o1/diamond.3.6,7:26.metrics.json
  em=0.409  f1=0.409  valid=165 of 198

outputs/runs.baselines/hotpotqa.deepseek-r1-distill-qwen-7b.naive_rag/test.3.6,8:56.metrics.json
  em=0.284  f1=0.382  valid=495 of 500

outputs/runs.baselines/hotpotqa.deepseek-r1-distill-qwen-7b.rag_agent/test.3.6,9:1.metrics.json
  em=0.064  f1=0.127  valid=491 of 500

outputs/runs.baselines/hotpotqa.deepseek-r1-distill-qwen-7b.search_o1/test.3.6,9:5.metrics.json
  em=0.086  f1=0.161  vali

In [ ]:
import json, glob

for f in sorted(glob.glob('outputs/**/*.metrics*.json', recursive=True)):
    if 'diamond.3.6,6:6' in f:
        continue
    with open(f) as fh:
        data = json.load(fh)
    overall = data['overall']
    print(f"{f}")
    print(f"  em={overall['em']:.3f}  f1={overall.get('f1', 0):.3f}  valid={overall['num_valid_answer']}")
    print()

outputs/gpqa.ds-qwen-7b.direct/diamond.3.6,6:24.metrics.json
  em=0.465  f1=0.465  valid=196 of 198

outputs/runs.baselines/gpqa.deepseek-r1-distill-qwen-7b.naive_rag/diamond.3.6,6:58.metrics.json
  em=0.500  f1=0.500  valid=188 of 198

outputs/runs.baselines/gpqa.deepseek-r1-distill-qwen-7b.rag_agent/diamond.3.6,7:12.metrics.backoff.json
  em=0.510  f1=0.510  valid=196 of 198

outputs/runs.baselines/gpqa.deepseek-r1-distill-qwen-7b.rag_agent/diamond.3.6,7:12.metrics.json
  em=0.439  f1=0.439  valid=167 of 198

outputs/runs.baselines/gpqa.deepseek-r1-distill-qwen-7b.search_o1/diamond.3.6,7:26.metrics.backoff.json
  em=0.470  f1=0.470  valid=196 of 198

outputs/runs.baselines/gpqa.deepseek-r1-distill-qwen-7b.search_o1/diamond.3.6,7:26.metrics.json
  em=0.409  f1=0.409  valid=165 of 198

outputs/runs.baselines/hotpotqa.deepseek-r1-distill-qwen-7b.naive_rag/test.3.6,8:56.metrics.json
  em=0.284  f1=0.382  valid=495 of 500

outputs/runs.baselines/hotpotqa.deepseek-r1-distill-qwen-7b.rag_ag